# Notebook C (P100 CUDA FIX) — Gated Theme-first + Binary Verifier (runs on Kaggle P100)
This notebook **forces a P100-compatible CUDA stack** by creating a **separate Python 3.11 + PyTorch CUDA 11.8** environment *inside the notebook* (micromamba),
then runs training/inference as a subprocess using that environment.

✅ Works on **Tesla P100 (sm_60)**  
✅ Keeps output small (saves **only best** checkpoint per fold)  
✅ Includes: fold-wise metrics, accepted-only theme macro-F1, temperature scaling, gradient accumulation, gating thresholds.


In [7]:
# 1) Create a P100-compatible environment (Python 3.11 + PyTorch CUDA 11.8)
# This is necessary because many Kaggle Python 3.12 images ship PyTorch CUDA builds that DO NOT support P100 (sm_60).
# We do NOT rely on the notebook kernel's torch.

!rm -rf /kaggle/working/py311_cu118 || true
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Create env with compatible scientific stack + HF
!./bin/micromamba create -y -p /kaggle/working/py311_cu118 \
  -c conda-forge -c pytorch -c nvidia \
  python=3.11 \
  pytorch=2.1.* torchvision torchaudio pytorch-cuda=11.8 \
  numpy=1.26.* scipy=1.11.* scikit-learn=1.4.* \
  pandas=2.2.* openpyxl \
  transformers=4.40.* accelerate=0.30.* datasets=2.20.* evaluate=0.4.* tokenizers sentencepiece

# Verify the env sees P100 CUDA
!/kaggle/working/py311_cu118/bin/python - <<'PY'
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu :", torch.cuda.get_device_name(0))
print("cap :", torch.cuda.get_device_capability(0))
PY


bin/micromamba
conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
pytorch/linux-64                                            Using cache
pytorch/noarch                                              Using cache
nvidia/linux-64                                             Using cache
nvidia/noarch                                               Using cache


Transaction

  Prefix: /kaggle/working/py311_cu118

  Updating specs:

   - python=3.11
   - pytorch=2.1
   - torchvision
   - torchaudio
   - pytorch-cuda=11.8
   - numpy=1.26
   - scipy=1.11
   - scikit-learn=1.4
   - pandas=2.2
   - openpyxl
   - transformers=4.40
   - accelerate=0.30
   - datasets=2.20
   - evaluate=0.4
   - tokenizers
   - sentencepiece


  Package                                      Version  Build                      Channel           Size
──────────────────────────────────────────────────────────────────────────────────

NameError: name 'PY' is not defined

In [8]:
# 2) Write training script (runs inside the py311_cu118 env)
import textwrap, os, json, pathlib

script = r'''
import os, re, json, random, math, shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

from transformers import AutoTokenizer, AutoModel
from torch.cuda.amp import autocast, GradScaler

# --------------------------
# Config
# --------------------------
class CFG:
    # Kaggle paths (change if your dataset is mounted elsewhere)
    train_path = os.environ.get("TRAIN_PATH", "/kaggle/input/climate-text-dataset/Human labelled_DTU.xlsx")
    test_path  = os.environ.get("TEST_PATH",  "/kaggle/input/climate-text-dataset/Master file_10k papers.xlsx")
    output_dir = os.environ.get("OUT_DIR", "/kaggle/working/notebook_C_p100_cuda/")

    model_name_theme = "microsoft/deberta-v3-base"
    model_name_bin   = "microsoft/deberta-v3-base"

    max_length = 192
    n_folds = 5

    epochs_theme = 6
    epochs_bin   = 4

    batch_size = 8
    grad_accum_steps = 2

    lr_theme = 2e-5
    lr_bin   = 2e-5
    weight_decay = 0.01
    max_grad_norm = 1.0
    early_stopping_patience = 2

    seed = 42
    num_workers = 2

    fp16 = True
    device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\\u00a0", " ")
    x = re.sub(r"\\s+", " ", x).strip()
    return x

def safe_mkdir(path):
    os.makedirs(path, exist_ok=True)

def compute_binary_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
    }

def tune_threshold_for_f1(y_true, y_prob):
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.05, 0.95, 91):
        f1m = f1_score(y_true, (y_prob >= thr).astype(int), average="macro", zero_division=0)
        if f1m > best_f1:
            best_thr, best_f1 = float(thr), float(f1m)
    return best_thr, best_f1

# Temperature scaling (binary)
def fit_temperature_binary(logits, y_true, max_steps=200, lr=0.05):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    logit_t = torch.tensor(logits, dtype=torch.float32, device=device)
    y = torch.tensor(y_true, dtype=torch.float32, device=device)
    log_T = torch.nn.Parameter(torch.zeros(1, device=device))
    opt = torch.optim.Adam([log_T], lr=lr)
    bce = torch.nn.BCEWithLogitsLoss()
    for _ in range(max_steps):
        opt.zero_grad(set_to_none=True)
        T = torch.exp(log_T).clamp(0.05, 20.0)
        loss = bce(logit_t / T, y)
        loss.backward()
        opt.step()
    return float(torch.exp(log_T).clamp(0.05, 20.0).detach().cpu().item())

def apply_temperature_binary(logits, T):
    return 1.0 / (1.0 + np.exp(-(logits / T)))

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }

def collate_pad(batch, pad_id):
    input_ids = [b["input_ids"] for b in batch]
    attention_mask = [b["attention_mask"] for b in batch]
    input_ids = nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
    attention_mask = nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    return {"input_ids": input_ids, "attention_mask": attention_mask}

class DebertaEncoder(nn.Module):
    def __init__(self, model_name, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.dropout(cls)

class ThemeFirstModel(nn.Module):
    def __init__(self, model_name, num_themes, dropout=0.2):
        super().__init__()
        self.backbone = DebertaEncoder(model_name, dropout=dropout)
        self.head = nn.Linear(self.backbone.hidden, num_themes)

    def forward(self, input_ids, attention_mask):
        x = self.backbone(input_ids, attention_mask)
        return self.head(x)

class BinaryModel(nn.Module):
    def __init__(self, model_name, dropout=0.2):
        super().__init__()
        self.backbone = DebertaEncoder(model_name, dropout=dropout)
        self.head = nn.Linear(self.backbone.hidden, 2)

    def forward(self, input_ids, attention_mask):
        x = self.backbone(input_ids, attention_mask)
        return self.head(x)

def make_multilabel_targets(df, K):
    Y = np.zeros((len(df), K), dtype=np.float32)
    for i in range(len(df)):
        if int(df.loc[i, "binary_label"]) == 1:
            t = int(df.loc[i, "theme_label"])
            if t >= 0:
                Y[i, t] = 1.0
    return Y

def theme_f1_accept_only(true_theme, pred_theme, y_true_bin):
    idx = np.where(y_true_bin == 1)[0]
    if len(idx) == 0:
        return float("nan")
    return float(f1_score(true_theme[idx], pred_theme[idx], average="macro", zero_division=0))

def load_data():
    # Training file (same layout you used earlier)
    train_df = pd.read_excel(CFG.train_path, skiprows=1)
    train_df.columns = [
        "Coder name","Article ID","Paper_Author/s","Paper title",
        "Year of publication","DOI","URL","Abstracts",
        "Accept/Reject","If Accept, identify theme"
    ]
    train_df = train_df[train_df["Accept/Reject"].isin(["Accept","Reject"])].copy()
    train_df["title"] = train_df["Paper title"].apply(clean_text)
    train_df["abstract"] = train_df["Abstracts"].apply(clean_text)
    train_df["text"] = (train_df["title"] + " [SEP] " + train_df["abstract"]).str.strip()
    train_df = train_df[train_df["text"].str.len() > 50].reset_index(drop=True)
    train_df["binary_label"] = (train_df["Accept/Reject"] == "Accept").astype(int)

    accepted_df = train_df[train_df["binary_label"] == 1].copy()
    unique_themes = accepted_df["If Accept, identify theme"].dropna().unique()
    theme_to_id = {t:i for i,t in enumerate(sorted(unique_themes))}
    id_to_theme = {i:t for t,i in theme_to_id.items()}

    train_df["theme_label"] = train_df["If Accept, identify theme"].map(theme_to_id)
    train_df["theme_label"] = train_df["theme_label"].fillna(-1).astype(int)

    test_df = pd.read_excel(CFG.test_path).copy()
    title_col = None
    for c in ["Article Title","Paper title","Title"]:
        if c in test_df.columns:
            title_col = c
            break
    test_df["title"] = "" if title_col is None else test_df[title_col].apply(clean_text)
    abs_col = "Abstract" if "Abstract" in test_df.columns else ("Abstracts" if "Abstracts" in test_df.columns else None)
    if abs_col is None:
        raise ValueError("Could not find Abstract column in prediction dataset.")
    test_df["abstract"] = test_df[abs_col].apply(clean_text)
    test_df["text"] = (test_df["title"] + " [SEP] " + test_df["abstract"]).str.strip()
    test_df = test_df[test_df["text"].str.len() > 50].reset_index(drop=True)

    return train_df, test_df, theme_to_id, id_to_theme

def train_theme_fold(fold, df, tokenizer, theme_to_id):
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    tr_idx, va_idx = list(skf.split(df, df["binary_label"]))[fold]
    tr = df.iloc[tr_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)

    K = len(theme_to_id)
    Y_tr = make_multilabel_targets(tr, K)

    ds_tr = TextDataset(tr["text"].tolist(), tokenizer, CFG.max_length)
    ds_va = TextDataset(va["text"].tolist(), tokenizer, CFG.max_length)

    ld_tr = DataLoader(ds_tr, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers,
                       collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id))
    ld_va = DataLoader(ds_va, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers,
                       collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id))

    model = ThemeFirstModel(CFG.model_name_theme, K, dropout=0.2).to(CFG.device)

    pos = Y_tr.sum(axis=0); neg = len(Y_tr) - pos
    pos_weight = torch.tensor((neg / np.clip(pos, 1, None)), dtype=torch.float32, device=CFG.device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr_theme, weight_decay=CFG.weight_decay)
    scaler = GradScaler(enabled=(CFG.fp16 and CFG.device=="cuda"))

    best_score = -1.0
    best = {"path": None, "temp": 1.0, "thr": 0.5}
    best_path = os.path.join(CFG.output_dir, f"best_theme_fold{fold}.pt")
    patience = 0
    log_rows = []

    for epoch in range(1, CFG.epochs_theme + 1):
        model.train()
        total = 0.0
        opt.zero_grad(set_to_none=True)

        for step, batch in enumerate(ld_tr):
            bsz = batch["input_ids"].size(0)
            s = step * CFG.batch_size
            y = torch.tensor(Y_tr[s:s+bsz], dtype=torch.float32, device=CFG.device)
            input_ids = batch["input_ids"].to(CFG.device)
            attention_mask = batch["attention_mask"].to(CFG.device)

            with autocast(enabled=(CFG.fp16 and CFG.device=="cuda")):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, y) / CFG.grad_accum_steps

            scaler.scale(loss).backward()
            total += loss.item() * bsz * CFG.grad_accum_steps

            if ((step+1) % CFG.grad_accum_steps == 0) or (step+1 == len(ld_tr)):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

        # Validation logits
        model.eval()
        all_logits = []
        with torch.no_grad():
            for batch in ld_va:
                input_ids = batch["input_ids"].to(CFG.device)
                attention_mask = batch["attention_mask"].to(CFG.device)
                all_logits.append(model(input_ids, attention_mask).float().cpu())
        logits = torch.cat(all_logits, 0).numpy()  # (N,K)

        accept_logit = logits.max(axis=1)
        y_true_bin = va["binary_label"].values.astype(int)

        T = fit_temperature_binary(accept_logit, y_true_bin)
        accept_prob = apply_temperature_binary(accept_logit, T)
        thr, _ = tune_threshold_for_f1(y_true_bin, accept_prob)
        bin_metrics = compute_binary_metrics(y_true_bin, accept_prob, thr)

        pred_theme = (1/(1+np.exp(-(logits/T)))).argmax(axis=1)
        theme_f1 = theme_f1_accept_only(va["theme_label"].values.astype(int), pred_theme, y_true_bin)

        score = bin_metrics["f1_macro"] + 0.25 * bin_metrics["pr_auc"]
        row = {
            "fold": fold, "epoch": epoch, "train_loss": float(total/len(tr)),
            **{f"theme_bin_{k}": v for k,v in bin_metrics.items()},
            "theme_f1_macro_accept_only": float(theme_f1),
            "theme_thr": float(thr), "theme_temp_T": float(T),
        }
        log_rows.append(row)
        print(f"[Theme] fold{fold} ep{epoch} loss{row['train_loss']:.4f} F1m{row['theme_bin_f1_macro']:.4f} PRAUC{row['theme_bin_pr_auc']:.4f} ThemeF1(acc){theme_f1:.4f} thr{thr:.2f} T{T:.2f}")

        if score > best_score:
            best_score = score
            best = {"path": best_path, "temp": float(T), "thr": float(thr)}
            patience = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience += 1
            if patience >= CFG.early_stopping_patience:
                break

    log_df = pd.DataFrame(log_rows)
    log_df.to_csv(os.path.join(CFG.output_dir, f"theme_fold{fold}_metrics.csv"), index=False)
    return best, log_df

def train_bin_fold(fold, df, tokenizer):
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    tr_idx, va_idx = list(skf.split(df, df["binary_label"]))[fold]
    tr = df.iloc[tr_idx].reset_index(drop=True)
    va = df.iloc[va_idx].reset_index(drop=True)

    y_tr = tr["binary_label"].values.astype(np.int64)

    ds_tr = TextDataset(tr["text"].tolist(), tokenizer, CFG.max_length)
    ds_va = TextDataset(va["text"].tolist(), tokenizer, CFG.max_length)
    ld_tr = DataLoader(ds_tr, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers,
                       collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id))
    ld_va = DataLoader(ds_va, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers,
                       collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id))

    model = BinaryModel(CFG.model_name_bin, dropout=0.2).to(CFG.device)

    counts = np.bincount(y_tr, minlength=2)
    w = counts.sum()/(2*np.clip(counts,1,None))
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=CFG.device))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr_bin, weight_decay=CFG.weight_decay)
    scaler = GradScaler(enabled=(CFG.fp16 and CFG.device=="cuda"))

    best_score = -1.0
    best = {"path": None, "temp": 1.0, "thr": 0.5}
    best_path = os.path.join(CFG.output_dir, f"best_bin_fold{fold}.pt")
    patience = 0
    log_rows = []

    for epoch in range(1, CFG.epochs_bin + 1):
        model.train()
        total = 0.0
        opt.zero_grad(set_to_none=True)

        for step, batch in enumerate(ld_tr):
            bsz = batch["input_ids"].size(0)
            s = step * CFG.batch_size
            labels = torch.tensor(y_tr[s:s+bsz], dtype=torch.long, device=CFG.device)
            input_ids = batch["input_ids"].to(CFG.device)
            attention_mask = batch["attention_mask"].to(CFG.device)

            with autocast(enabled=(CFG.fp16 and CFG.device=="cuda")):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels) / CFG.grad_accum_steps

            scaler.scale(loss).backward()
            total += loss.item() * bsz * CFG.grad_accum_steps

            if ((step+1) % CFG.grad_accum_steps == 0) or (step+1 == len(ld_tr)):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

        # Validation logits
        model.eval()
        all_logits = []
        with torch.no_grad():
            for batch in ld_va:
                input_ids = batch["input_ids"].to(CFG.device)
                attention_mask = batch["attention_mask"].to(CFG.device)
                all_logits.append(model(input_ids, attention_mask).float().cpu())
        logits = torch.cat(all_logits, 0).numpy()  # (N,2)

        pos_logit = logits[:,1] - logits[:,0]
        y_true = va["binary_label"].values.astype(int)

        T = fit_temperature_binary(pos_logit, y_true)
        prob = apply_temperature_binary(pos_logit, T)
        thr, _ = tune_threshold_for_f1(y_true, prob)
        bin_metrics = compute_binary_metrics(y_true, prob, thr)

        score = bin_metrics["f1_macro"] + 0.25 * bin_metrics["pr_auc"]
        row = {
            "fold": fold, "epoch": epoch, "train_loss": float(total/len(tr)),
            **{f"bin_{k}": v for k,v in bin_metrics.items()},
            "thr": float(thr), "temp_T": float(T),
        }
        log_rows.append(row)
        print(f"[Bin ] fold{fold} ep{epoch} loss{row['train_loss']:.4f} F1m{row['bin_f1_macro']:.4f} PRAUC{row['bin_pr_auc']:.4f} thr{thr:.2f} T{T:.2f}")

        if score > best_score:
            best_score = score
            best = {"path": best_path, "temp": float(T), "thr": float(thr)}
            patience = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience += 1
            if patience >= CFG.early_stopping_patience:
                break

    log_df = pd.DataFrame(log_rows)
    log_df.to_csv(os.path.join(CFG.output_dir, f"bin_fold{fold}_metrics.csv"), index=False)
    return best, log_df

def predict_theme_probs(models, tokenizer, texts, K):
    ds = TextDataset(texts, tokenizer, CFG.max_length)
    loader = DataLoader(ds, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers,
                        collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id))
    probs_f=[]
    for bm in models:
        m = ThemeFirstModel(CFG.model_name_theme, K, dropout=0.0).to(CFG.device)
        m.load_state_dict(torch.load(bm["path"], map_location=CFG.device))
        m.eval()
        all_logits=[]
        with torch.no_grad():
            for batch in loader:
                input_ids=batch["input_ids"].to(CFG.device)
                attention_mask=batch["attention_mask"].to(CFG.device)
                all_logits.append(m(input_ids, attention_mask).float().cpu())
        logits=torch.cat(all_logits,0).numpy()
        T=bm["temp"]
        probs=1/(1+np.exp(-(logits/T)))
        probs_f.append(probs)
        del m
        torch.cuda.empty_cache()
    return np.mean(probs_f, axis=0)

def predict_bin_prob(models, tokenizer, texts):
    ds = TextDataset(texts, tokenizer, CFG.max_length)
    loader = DataLoader(ds, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers,
                        collate_fn=lambda b: collate_pad(b, tokenizer.pad_token_id))
    probs_f=[]
    for bm in models:
        m = BinaryModel(CFG.model_name_bin, dropout=0.0).to(CFG.device)
        m.load_state_dict(torch.load(bm["path"], map_location=CFG.device))
        m.eval()
        all_logits=[]
        with torch.no_grad():
            for batch in loader:
                input_ids=batch["input_ids"].to(CFG.device)
                attention_mask=batch["attention_mask"].to(CFG.device)
                all_logits.append(m(input_ids, attention_mask).float().cpu())
        logits=torch.cat(all_logits,0).numpy()
        pos_logit = logits[:,1] - logits[:,0]
        prob = apply_temperature_binary(pos_logit, bm["temp"])
        probs_f.append(prob)
        del m
        torch.cuda.empty_cache()
    return np.mean(probs_f, axis=0)

def main():
    seed_everything(CFG.seed)
    safe_mkdir(CFG.output_dir)
    print("Device:", CFG.device)

    train_df, test_df, theme_to_id, id_to_theme = load_data()
    K = len(theme_to_id)
    print("Train:", len(train_df), "Test:", len(test_df), "Themes:", K)
    print("Binary:", train_df["binary_label"].value_counts().to_dict())

    tokenizer_theme = AutoTokenizer.from_pretrained(CFG.model_name_theme, use_fast=False)
    tokenizer_bin   = AutoTokenizer.from_pretrained(CFG.model_name_bin, use_fast=False)

    theme_bests=[]; bin_bests=[]
    logs=[]
    for fold in range(CFG.n_folds):
        tb, tlog = train_theme_fold(fold, train_df, tokenizer_theme, theme_to_id)
        bb, blog = train_bin_fold(fold, train_df, tokenizer_bin)
        theme_bests.append(tb); bin_bests.append(bb)
        logs.append(tlog.assign(model="theme"))
        logs.append(blog.assign(model="binary"))

    pd.concat(logs, ignore_index=True).to_csv(os.path.join(CFG.output_dir,"all_metrics.csv"), index=False)
    with open(os.path.join(CFG.output_dir,"theme_to_id.json"),"w") as f: json.dump(theme_to_id,f,indent=2)
    with open(os.path.join(CFG.output_dir,"id_to_theme.json"),"w") as f: json.dump(id_to_theme,f,indent=2)
    with open(os.path.join(CFG.output_dir,"theme_best.json"),"w") as f: json.dump(theme_bests,f,indent=2)
    with open(os.path.join(CFG.output_dir,"bin_best.json"),"w") as f: json.dump(bin_bests,f,indent=2)

    # Tune gating thresholds on fold-0 validation only (speed)
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    tr_idx, va_idx = list(skf.split(train_df, train_df["binary_label"]))[0]
    va = train_df.iloc[va_idx].reset_index(drop=True)

    tp = predict_theme_probs(theme_bests, tokenizer_theme, va["text"].tolist(), K)
    mprob = tp.max(axis=1)
    bp = predict_bin_prob(bin_bests, tokenizer_bin, va["text"].tolist())
    y_true = va["binary_label"].values.astype(int)

    best=None
    for tau_low in np.linspace(0.05, 0.40, 8):
        for tau_high in np.linspace(0.55, 0.90, 8):
            if tau_low >= tau_high: 
                continue
            pred = np.zeros_like(y_true)
            pred[mprob >= tau_high] = 1
            mid = (mprob >= tau_low) & (mprob < tau_high)
            pred[mid] = (bp[mid] >= 0.5).astype(int)

            f1m = f1_score(y_true, pred, average="macro", zero_division=0)
            pr = average_precision_score(y_true, np.maximum(mprob, bp))
            score = f1m + 0.25*pr
            if best is None or score > best["score"]:
                best = {"tau_low": float(tau_low), "tau_high": float(tau_high), "f1_macro": float(f1m), "pr_auc": float(pr), "score": float(score)}
    print("Best gating:", best)
    with open(os.path.join(CFG.output_dir,"gating_thresholds.json"),"w") as f: json.dump(best,f,indent=2)

    # Inference
    tp = predict_theme_probs(theme_bests, tokenizer_theme, test_df["text"].tolist(), K)
    mprob = tp.max(axis=1)
    bp = predict_bin_prob(bin_bests, tokenizer_bin, test_df["text"].tolist())

    tau_low, tau_high = best["tau_low"], best["tau_high"]
    pred = np.zeros(len(test_df), dtype=int)
    pred[mprob >= tau_high] = 1
    mid = (mprob >= tau_low) & (mprob < tau_high)
    pred[mid] = (bp[mid] >= 0.5).astype(int)

    theme_id = tp.argmax(axis=1)
    theme = [id_to_theme[int(i)] for i in theme_id]
    theme = [theme[i] if pred[i]==1 else "" for i in range(len(theme))]

    out = test_df.copy()
    out["Prediction_Accept_Reject"] = np.where(pred==1, "Accept", "Reject")
    out["Accept_Confidence"] = np.maximum(mprob, bp)
    out["Predicted_Theme"] = theme

    out_path = os.path.join(CFG.output_dir, "predictions.csv")
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)

if __name__ == "__main__":
    main()
'''
pathlib.Path("/kaggle/working/train_notebook_c.py").write_text(script)
print("Wrote /kaggle/working/train_notebook_c.py")


Wrote /kaggle/working/train_notebook_c.py


In [9]:
# 3) Run training + inference inside the P100 CUDA env
# If your dataset paths differ, set env vars TRAIN_PATH and TEST_PATH here.

# Example (if using Kaggle dataset mount):
# %env TRAIN_PATH=/kaggle/input/climate-text-dataset/Human labelled_DTU.xlsx
# %env TEST_PATH=/kaggle/input/climate-text-dataset/Master file_10k papers.xlsx

!/kaggle/working/py311_cu118/bin/python /kaggle/working/train_notebook_c.py


Device: cuda
/kaggle/working/py311_cu118/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Train: 1720 Test: 10176 Themes: 12
Binary: {0: 1521, 1: 199}
/kaggle/working/py311_cu118/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/kaggle/working/py311_cu118/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/kaggle/working/py311_cu118/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and 

In [10]:
# 4) Outputs
# Best checkpoints + logs + predictions are saved under:
# /kaggle/working/notebook_C_p100_cuda/

!ls -lh /kaggle/working/notebook_C_p100_cuda/ | head -n 50

total 7.0G
-rw-r--r-- 1 root root 6.1K Feb 21 15:15 all_metrics.csv
-rw-r--r-- 1 root root 702M Feb 21 14:52 best_bin_fold0.pt
-rw-r--r-- 1 root root 702M Feb 21 14:58 best_bin_fold1.pt
-rw-r--r-- 1 root root 702M Feb 21 15:04 best_bin_fold2.pt
-rw-r--r-- 1 root root 702M Feb 21 15:10 best_bin_fold3.pt
-rw-r--r-- 1 root root 702M Feb 21 15:15 best_bin_fold4.pt
-rw-r--r-- 1 root root 702M Feb 21 14:50 best_theme_fold0.pt
-rw-r--r-- 1 root root 702M Feb 21 14:55 best_theme_fold1.pt
-rw-r--r-- 1 root root 702M Feb 21 15:00 best_theme_fold2.pt
-rw-r--r-- 1 root root 702M Feb 21 15:06 best_theme_fold3.pt
-rw-r--r-- 1 root root 702M Feb 21 15:11 best_theme_fold4.pt
-rw-r--r-- 1 root root  665 Feb 21 15:15 bin_best.json
-rw-r--r-- 1 root root  675 Feb 21 14:54 bin_fold0_metrics.csv
-rw-r--r-- 1 root root  629 Feb 21 14:59 bin_fold1_metrics.csv
-rw-r--r-- 1 root root  690 Feb 21 15:04 bin_fold2_metrics.csv
-rw-r--r-- 1 root root  680 Feb 21 15:10 bin_fold3_metrics.csv
-rw-r--r-- 1 root root  6

In [12]:
import pandas as pd
df=pd.read_csv("/kaggle/working/notebook_C_p100_cuda/predictions.csv")
df

,ID_OLD,ID_New,Authors,Article Title,Publication Year,DOI,DOI Link,Abstract,title,abstract,text,Prediction_Accept_Reject,Accept_Confidence,Predicted_Theme
0,OA_3712,OA_3712,Cornelia Marietje Aneke Wattimena,NaN,2022,NaN,https://doi.org/10.32832/abdidos.v6i3.1321,The current use of forests should be directed ...,NaN,The current use of forests should be directed ...,[SEP] The current use of forests should be dir...,Reject,0.104460,NaN
1,WoS_1385,WoS_1385,"Weatherly, C; Doherty, FC","It ' s one thing after another, after another...",2025,10.1016/j.jrurstud.2025.103573,http://dx.doi.org/10.1016/j.jrurstud.2025.103573,Because of their closeness to and dependence o...,"It ' s one thing after another, after another:...",Because of their closeness to and dependence o...,"It ' s one thing after another, after another:...",Reject,0.107571,NaN
2,Scopus_5109,Scopus_5109,"L., Cameron, Laura; I.J., Mauro, Ian J.; K., S...","""A Return to and of the Land"": Indigenous Know...",2021,10.2993/0278-0771-41.3.368,NaN,While research on Indigenous knowledges on cli...,"""A Return to and of the Land"": Indigenous Know...",While research on Indigenous knowledges on cli...,"""A Return to and of the Land"": Indigenous Know...",Reject,0.111577,NaN
3,Scopus_4859,Scopus_4859,"A.K., Menzies, Allyson K.; E., Bowlesa, E.; M....","""I see my culture starting to disappear"": Anis...",2022,10.1139/facets-2021-0066,NaN,Climate change disproportionately affects Indi...,"""I see my culture starting to disappear"": Anis...",Climate change disproportionately affects Indi...,"""I see my culture starting to disappear"": Anis...",Reject,0.105532,NaN
4,Scopus_1176,Scopus_1176,"A., Gellu, Ashok; D., Sharma, Damini; S., Rani...","""Impact of Climate Change on Coastal Cities: A...",2025,10.53555/jab.v11i2.211,NaN,This research article aims to establish the ef...,"""Impact of Climate Change on Coastal Cities: A...",This research article aims to establish the ef...,"""Impact of Climate Change on Coastal Cities: A...",Reject,0.105238,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10171,OA_4235,OA_4235,Сырбек П.Н.|Бимендиева Л.А.|Мурабилдаева Р.А.|...,ҚАЗАҚСТАН РЕСПУБЛИКАСЫНДАҒЫ АЗЫҚ-ТҮЛІК ЖҮЙЕСІН...,2024,NaN,https://doi.org/10.52260/2304-7216.2024.3(56).4,Food systems are at the heart of the 2030 Agen...,ҚАЗАҚСТАН РЕСПУБЛИКАСЫНДАҒЫ АЗЫҚ-ТҮЛІК ЖҮЙЕСІН...,Food systems are at the heart of the 2030 Agen...,ҚАЗАҚСТАН РЕСПУБЛИКАСЫНДАҒЫ АЗЫҚ-ТҮЛІК ЖҮЙЕСІН...,Reject,0.112315,NaN
10172,OA_3078,OA_3078,А. Оразаліна|Г.А. Кадирова,СУЧАСНИЙ СТАН БАНКІВСЬКОЇ СИСТЕМИ РЕСПУБЛІКИ К...,2023,NaN,https://doi.org/10.32782/2222-5099.2023.9.21,"The Sustainable Development Goal ""Ending hunge...",СУЧАСНИЙ СТАН БАНКІВСЬКОЇ СИСТЕМИ РЕСПУБЛІКИ К...,"The Sustainable Development Goal ""Ending hunge...",СУЧАСНИЙ СТАН БАНКІВСЬКОЇ СИСТЕМИ РЕСПУБЛІКИ К...,Reject,0.107708,NaN
10173,OA_4940,OA_4940,Samvel AVETISYAN,Պարենային անվտանգության ոլորտում ԵԱՏՄ երկրների...,2024,NaN,https://doi.org/10.52174/2579-2989_2024.1-60,"At present, the world is very concerned. This ...",Պարենային անվտանգության ոլորտում ԵԱՏՄ երկրների...,"At present, the world is very concerned. This ...",Պարենային անվտանգության ոլորտում ԵԱՏՄ երկրների...,Reject,0.138946,NaN
10174,OA_4085,OA_4085,د المعتصم بالله البحراوي د رامي السعدني,آثار تغير المناخ على الاقتصاد الأزرق في مصر,2023,NaN,https://doi.org/10.21608/mjle.2023.314883,"The term ""blue economy"" was widely adopted as ...",آثار تغير المناخ على الاقتصاد الأزرق في مصر,"The term ""blue economy"" was widely adopted as ...",آثار تغير المناخ على الاقتصاد الأزرق في مصر [S...,Reject,0.121266,NaN


In [13]:
df1=pd.read_csv("/kaggle/working/notebook_C_p100_cuda/all_metrics.csv")
df1

,fold,epoch,train_loss,theme_bin_accuracy,theme_bin_precision_macro,theme_bin_recall_macro,theme_bin_f1_macro,theme_bin_pr_auc,theme_f1_macro_accept_only,theme_thr,theme_temp_T,model,bin_accuracy,bin_precision_macro,bin_recall_macro,bin_f1_macro,bin_pr_auc,thr,temp_T
0,0,1,1.484131,0.741279,0.504191,0.506250,0.499861,0.128406,0.016529,0.12,0.817435,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,2.232258,0.883721,0.441860,0.500000,0.469136,0.149211,0.062069,0.17,1.134780,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,3,2.755950,0.883721,0.441860,0.500000,0.469136,0.121494,0.000000,0.14,1.139947,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,1,0.643014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.837209,0.582353,0.571382,0.576019,0.166921,0.13,2.166393
4,0,2,0.782926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.755814,0.594313,0.677303,0.601654,0.270595,0.12,1.016153
5,0,3,0.647913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.837209,0.582353,0.571382,0.576019,0.217012,0.12,1.759979
6,0,4,0.722357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.802326,0.545813,0.551645,0.548173,0.198534,0.12,1.992146
7,1,1,1.585103,0.767442,0.530516,0.542763,0.531973,0.177796,0.016529,0.12,0.950721,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,2,2.388818,0.883721,0.441860,0.500000,0.469136,0.154042,0.059649,0.12,1.241832,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,3,2.440440,0.848837,0.544444,0.523684,0.525265,0.176131,0.059649,0.12,1.002529,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df2=pd.read_csv("/kaggle/working/notebook_C_p100_cuda/all_metrics.csv")
df2

,fold,epoch,train_loss,theme_bin_accuracy,theme_bin_precision_macro,theme_bin_recall_macro,theme_bin_f1_macro,theme_bin_pr_auc,theme_f1_macro_accept_only,theme_thr,theme_temp_T,model,bin_accuracy,bin_precision_macro,bin_recall_macro,bin_f1_macro,bin_pr_auc,thr,temp_T
0,0,1,1.484131,0.741279,0.504191,0.506250,0.499861,0.128406,0.016529,0.12,0.817435,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,2.232258,0.883721,0.441860,0.500000,0.469136,0.149211,0.062069,0.17,1.134780,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,3,2.755950,0.883721,0.441860,0.500000,0.469136,0.121494,0.000000,0.14,1.139947,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,1,0.643014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.837209,0.582353,0.571382,0.576019,0.166921,0.13,2.166393
4,0,2,0.782926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.755814,0.594313,0.677303,0.601654,0.270595,0.12,1.016153
5,0,3,0.647913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.837209,0.582353,0.571382,0.576019,0.217012,0.12,1.759979
6,0,4,0.722357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binary,0.802326,0.545813,0.551645,0.548173,0.198534,0.12,1.992146
7,1,1,1.585103,0.767442,0.530516,0.542763,0.531973,0.177796,0.016529,0.12,0.950721,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,2,2.388818,0.883721,0.441860,0.500000,0.469136,0.154042,0.059649,0.12,1.241832,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,3,2.440440,0.848837,0.544444,0.523684,0.525265,0.176131,0.059649,0.12,1.002529,theme,NaN,NaN,NaN,NaN,NaN,NaN,NaN
